# 🧬 ESMC Analysis Pipeline - VM Version

Production pipeline for Azure ML, AWS SageMaker, or local servers.

**Steps:**
1. FASTA Cleaning
2. Embedding Generation
3. Entropy Analysis
4. Logits Analysis
5. Export Results

---

In [ ]:
# ============================================================
# CONFIGURATION - Edit these paths
# ============================================================

# Input/Output paths
FASTA_INPUT = "data/sample_data/sample.fasta"  # Your FASTA file
SEQUENCES_CSV = "data/sequences.csv"
METADATA_CSV = "data/metadata.csv"
EMBEDDINGS_PT = "data/embeddings.pt"
OUTPUT_DIR = "results"

# Model configuration
HF_TOKEN = ""  # Your HuggingFace token (or set HF_TOKEN env var)
MODEL_NAME = "esmc_600m"  # or "esmc_300m"

# Analysis configuration
RESIDUES_OF_INTEREST = {
    0: "Pos 1",
    10: "Pos 11",
    20: "Pos 21",
    50: "Pos 51",
    100: "Pos 101",
}

print("📋 Configuration loaded")

In [ ]:
# ============================================================
# SETUP
# ============================================================

import os
import sys
from pathlib import Path

# Add src to path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Imports
import pandas as pd
import torch
from tqdm.auto import tqdm

# Verify imports
from src.embedding import process_fasta_files, load_esmc_model, embed_sequences, save_embeddings
from src.analysis import analyze_entropy, entropy_summary, analyze_residues, plot_heatmap, AA_VOCAB

# Create output directory
Path(OUTPUT_DIR).mkdir(exist_ok=True)

# Get token from env if not set
if not HF_TOKEN:
    HF_TOKEN = os.getenv("HF_TOKEN", "")

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ Pipeline packages loaded")

---
## Step 1: FASTA Cleaning

In [ ]:
# ============================================================
# STEP 1: FASTA CLEANING
# ============================================================

print(f"🔄 Processing: {FASTA_INPUT}")

sequences_df, metadata_df = process_fasta_files(FASTA_INPUT)

# Save
sequences_df.to_csv(SEQUENCES_CSV, index=False)
metadata_df.to_csv(METADATA_CSV, index=False)

print(f"✅ Processed {len(sequences_df)} sequences")
print(f"   Saved: {SEQUENCES_CSV}, {METADATA_CSV}")
sequences_df.head()

---
## Step 2: Embedding Generation

In [ ]:
# ============================================================
# STEP 2: EMBEDDING GENERATION
# ============================================================

if not HF_TOKEN:
    raise ValueError("Set HF_TOKEN in configuration or environment")

print(f"🔄 Loading model: {MODEL_NAME}")
model = load_esmc_model(HF_TOKEN, MODEL_NAME)
print(f"✅ Model loaded on {DEVICE}")

In [ ]:
# Generate embeddings with progress bar
print(f"🔄 Embedding {len(sequences_df)} sequences...")

pbar = tqdm(total=len(sequences_df), desc="Embedding")

def update_progress(current, total):
    pbar.n = current
    pbar.refresh()

embedding_results = embed_sequences(
    model, sequences_df,
    return_embeddings=True,
    return_logits=True,
    progress_callback=update_progress
)

pbar.close()

# Save
save_embeddings(embedding_results, EMBEDDINGS_PT)
print(f"✅ Saved: {EMBEDDINGS_PT}")

# Report errors
if embedding_results.get("errors"):
    print(f"⚠️ {len(embedding_results['errors'])} errors:")
    for seq_id, err in embedding_results["errors"][:5]:
        print(f"   {seq_id}: {err}")

---
## Step 3: Entropy Analysis

In [ ]:
# ============================================================
# STEP 3: ENTROPY ANALYSIS
# ============================================================

# Load embeddings if needed
if 'embedding_results' not in dir() or embedding_results is None:
    print(f"🔄 Loading embeddings from {EMBEDDINGS_PT}")
    embedding_results = torch.load(EMBEDDINGS_PT, weights_only=False)

print("🔄 Calculating entropy...")
entropy_results = analyze_entropy(
    embedding_results,
    base="e",
    constrained_percentile=10.0,
    flexible_percentile=90.0
)

entropy_df = entropy_summary(entropy_results)
entropy_df.to_csv(f"{OUTPUT_DIR}/entropy_summary.csv", index=False)

print(f"✅ Analyzed {len(entropy_df)} sequences")
print(f"📊 Global mean entropy: {entropy_results['global_mean']:.3f}")
entropy_df

In [ ]:
# Visualize entropy profile
import matplotlib.pyplot as plt

if len(entropy_results["entropy"]) > 0:
    vals = entropy_results["entropy"][0].float().numpy()
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(vals, alpha=0.8, linewidth=1.5)
    ax.set_xlabel("Residue Position")
    ax.set_ylabel("Entropy (nats)")
    ax.set_title(f"Entropy Profile: {entropy_results['sequence_id'][0]}")
    
    # Highlight regions
    constrained = entropy_results["constrained_positions"][0].long().numpy()
    flexible = entropy_results["flexible_positions"][0].long().numpy()
    
    ax.scatter(constrained, vals[constrained], c="blue", s=15, alpha=0.6, label="Constrained")
    ax.scatter(flexible, vals[flexible], c="red", s=15, alpha=0.6, label="Flexible")
    
    ax.axhline(y=entropy_results['global_mean'], color='gray', linestyle='--', alpha=0.5, label='Global mean')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/entropy_profile.png", dpi=150)
    plt.show()
    print(f"✅ Saved: {OUTPUT_DIR}/entropy_profile.png")

---
## Step 4: Logits Analysis

In [ ]:
# ============================================================
# STEP 4: LOGITS ANALYSIS
# ============================================================

print(f"🔄 Analyzing {len(RESIDUES_OF_INTEREST)} residues of interest...")

logits_analysis = analyze_residues(
    embedding_results,
    residues_of_interest=RESIDUES_OF_INTEREST,
    pool_method="mean",
    scale_method="minmax"
)

# Save probabilities
logits_analysis["probs"].to_csv(f"{OUTPUT_DIR}/logits_probs.csv")

print("✅ Analysis complete")
print(f"   Saved: {OUTPUT_DIR}/logits_probs.csv")
logits_analysis["probs"]

In [ ]:
# Generate heatmap
plot_heatmap(
    logits_analysis["scaled_logits"],
    row_labels=logits_analysis["residue_labels"],
    col_labels=AA_VOCAB,
    title="Amino Acid Propensity Heatmap",
    figsize=(14, 6),
    cmap="coolwarm",
    save_path=f"{OUTPUT_DIR}/heatmap.png"
)
print(f"✅ Saved: {OUTPUT_DIR}/heatmap.png")

---
## Step 5: Summary & Export

In [ ]:
# ============================================================
# STEP 5: SUMMARY
# ============================================================

print("\n" + "="*60)
print("📊 PIPELINE SUMMARY")
print("="*60)
print(f"\n✅ Input: {FASTA_INPUT}")
print(f"✅ Sequences processed: {len(sequences_df)}")
print(f"✅ Embeddings generated: {len(embedding_results['sequence_id'])}")
if embedding_results.get('errors'):
    print(f"⚠️ Errors: {len(embedding_results['errors'])}")
print(f"\n📁 Output files in '{OUTPUT_DIR}/':")

for f in Path(OUTPUT_DIR).glob("*"):
    size = f.stat().st_size / 1024
    print(f"   • {f.name} ({size:.1f} KB)")

print("\n🎉 Pipeline complete!")

In [ ]:
# Optional: Create zip archive
import shutil

archive_name = "pipeline_results"
shutil.make_archive(archive_name, "zip", OUTPUT_DIR)
print(f"✅ Created: {archive_name}.zip")